# Tutorial 4 — Complete AEGISS Workflow

This notebook ties together everything from the previous three tutorials into a
single end-to-end run of the AEGISS active space selection pipeline.

**What you will do**
1. Build a molecule and run a mean-field calculation
2. (Optional) Localise the orbitals
3. Configure the four AEGISS objects: `DMRGConfig`, `EntropyConfig`, `ProjectionConfig`, `OutputConfig`
4. Call `select_aegiss` — the single function that runs DMRG, entropy screening, and AO projection
5. Interpret the result and inspect saved artefacts
6. Explore threshold sensitivity and multi-group AO labels

**Working example**: benzene (C₆H₆) with CAS(6e, 6o) targeting the π system.
The correct answer is well known — six π MOs — so you can immediately verify the pipeline.

**Prerequisites**: `block2`, PySCF, NumPy.

## 1. Imports

In [ ]:
import numpy as np

# Four config dataclasses — one for each pipeline stage
from aegiss import DMRGConfig, EntropyConfig, OutputConfig, ProjectionConfig

# Top-level pipeline function
from aegiss.active_space.aegiss import select_aegiss

# Molecule and mean-field helpers
from aegiss.initial_guess.scf import build_mol, run_scf

# Optional: orbital localization
from aegiss.localization.orbital_localization import localize

## 2. Build the molecule

We use benzene in the cc-pVDZ basis — small enough for a fast DMRG run, large enough
to show meaningful entropy variation across the π and σ spaces.

In [ ]:
benzene_geom = """
C   1.40272  0.00000  0.00000
C   0.70136  1.21451  0.00000
C  -0.70136  1.21451  0.00000
C  -1.40272  0.00000  0.00000
C  -0.70136 -1.21451  0.00000
C   0.70136 -1.21451  0.00000
H   2.49118  0.00000  0.00000
H   1.24559  2.15720  0.00000
H  -1.24559  2.15720  0.00000
H  -2.49118  0.00000  0.00000
H  -1.24559 -2.15720  0.00000
H   1.24559 -2.15720  0.00000
"""

mol = build_mol(
    atom=benzene_geom,
    basis="ccpvdz",
    symmetry=False,
    verbose=0,
)
print(f"Atoms      : {mol.natm}")
print(f"AOs        : {mol.nao}")
print(f"Electrons  : {mol.nelectron}")

## 3. Mean-field calculation

In [ ]:
mf = run_scf(mol, method="RHF")

n_occ = int(np.sum(mf.mo_occ > 0))
print(f"RHF energy : {mf.e_tot:.8f} Ha")
print(f"MOs        : {mf.mo_coeff.shape[1]}  (occ: {n_occ})")

## 4. (Optional) Orbital localization

Localization is not required, but it can improve physical interpretability of
the entropy landscape — particularly for transition-metal systems where canonical
MOs are delocalized over the whole ligand field.

For benzene we skip localization: the canonical π MOs are already well separated
and the `C 2pz` AO projection correctly identifies them from canonical orbitals.

To use localized orbitals, replace `mf.mo_coeff` before calling `select_aegiss`:

```python
# Boys localization of the occupied space
mf.mo_coeff = localize("boys", mol, mf)

# Pipek–Mezey (preserves σ/π separation)
mf.mo_coeff = localize("pm", mol, mf)

# Natural Orbitals from MP2
mf.mo_coeff = localize("nos", mf, correlated_method="mp2")
```

See [01 — SCF & Localization](01_scf_and_localization.ipynb) for a detailed
comparison of all localization schemes.

## 5. Configure the four AEGISS objects

The full pipeline is driven by four frozen dataclasses.  Configure them once;
pass them all to `select_aegiss`.

### 5a. DMRGConfig — CAS and solver settings

| Field | Meaning |
|---|---|
| `ncas` | Number of active orbitals |
| `nelecas` | Active electrons (total int or α/β tuple) |
| `init_bond_dim` | Starting bond dimension; grows during sweeps |
| `symm_type` | `'SZ'` (required for entropies) or `'SU2'` (faster, no entropies) |
| `verbosity` | block2 output level (0 = silent) |

In [ ]:
# CAS(6e, 6o) — the benzene π space
dmrg_config = DMRGConfig(
    ncas=6,
    nelecas=6,
    init_bond_dim=20,
    symm_type="SZ",   # must be 'SZ' to obtain single-orbital entropies
    verbosity=0,
)
print(dmrg_config)

### 5b. EntropyConfig — entropy-based orbital screening

| Field | Default | Meaning |
|---|---|---|
| `fraction` | `0.1` | Threshold value (interpretation depends on `fraction_type`) |
| `fraction_type` | `'r'` | `'r'`: keep if $S_1 \geq$ `fraction × max(S_1)`; `'a'`: keep if $S_1 \geq$ `fraction` |
| `ranking` | `'subset'` | Apply threshold independently per subspace (`'subset'`) or jointly (`'global'`) |

In [ ]:
entropy_config = EntropyConfig(
    fraction=0.1,         # keep orbitals with S1 >= 10% of the maximum S1
    fraction_type="r",
    ranking="subset",     # threshold applied independently to occ and virt
)
print(entropy_config)

### 5c. ProjectionConfig — AO projection-based screening

| Field | Default | Meaning |
|---|---|---|
| `ao_labels` | — | AO groups to project onto (PySCF `ao_labels` strings) |
| `basis` | `'minao'` | Minimal basis used for the projection |
| `contribution` | `0.5` | Weight threshold (or per-group list) |
| `contribution_type` | `'a'` | `'a'` = absolute weight cutoff; `'r'` = relative to per-group maximum |
| `top_n` | `None` | Select the top-N orbitals per group instead of thresholding |
| `ranking` | `'subset'` | Apply per subspace (`'subset'`) or on the combined pool (`'global'`) |

For benzene we target the π system with `"C 2pz"`.  The six π MOs each have a
large weight on the carbon 2p_z AOs and should all pass the threshold.

In [ ]:
# Verify the AO label matches before building the config
pz_idx = mol.search_ao_label("C 2pz")
print(f"AOs matching 'C 2pz': {pz_idx}")
for i in pz_idx:
    print(f"  [{i:>2d}]  {mol.ao_labels()[i]}")

In [ ]:
projection_config = ProjectionConfig(
    ao_labels=["C 2pz"],    # the π AO group
    basis="minao",
    contribution=0.3,        # keep orbitals with weight >= 0.3 (absolute)
    contribution_type="a",
)
print(projection_config)

### 5d. OutputConfig — files and plots

| Field | Default | Meaning |
|---|---|---|
| `saving_folder` | `'./aegiss_output/'` | Directory for cube files and PDFs |
| `saving_prefix` | `'aegiss_selection'` | Filename prefix |
| `save_orb` | `True` | Write cube files for the entropy-screened and final orbitals |
| `save_plots` | `False` | Write diagnostic PDFs (SOE heatmap, TH plot, MI heatmap, entanglement diagram) |

In [ ]:
import tempfile, os

output_dir = os.path.join(tempfile.gettempdir(), "aegiss_benzene")
os.makedirs(output_dir, exist_ok=True)

output_config = OutputConfig(
    saving_folder=output_dir,
    saving_prefix="benzene_pi",
    save_orb=True,
    save_plots=True,
)
print(output_config)
print(f"\nOutput will be written to: {output_dir}")

## 6. Run the full AEGISS pipeline

`select_aegiss` orchestrates the entire workflow in one call:

1. Identifies the CAS orbital subspaces from `dmrg_config`
2. Runs the DMRG calculation and extracts single-orbital entropies
3. Applies the entropy threshold (`entropy_config`) → `screened_occ`, `screened_virt`
4. Projects the surviving orbitals onto the AO groups (`projection_config`) → final active space
5. Saves cube files and diagnostic PDFs if requested (`output_config`)

**Returns**: a sorted `np.ndarray` of integer orbital indices.

In [ ]:
active_indices = select_aegiss(
    mol=mol,
    mf=mf,
    dmrg_config=dmrg_config,
    entropy_config=entropy_config,
    projection_config=projection_config,
    output_config=output_config,
)

print(f"Active orbital indices : {active_indices}")
print(f"Active space size      : {len(active_indices)} orbitals")

## 7. Interpret the result

For benzene CAS(6e, 6o) with `"C 2pz"` projection we expect **six π MOs** —
the three bonding (occupied) and three antibonding (virtual) π orbitals.
These are the orbitals responsible for aromaticity and strongly correlated
character of the π system.

In [ ]:
# Separate occupied and virtual
n_occ = int(np.sum(mf.mo_occ > 0))
active_occ  = active_indices[active_indices < n_occ]
active_virt = active_indices[active_indices >= n_occ]

print(f"Total MOs             : {mf.mo_coeff.shape[1]}  (occ: {n_occ})")
print(f"CAS size              : {dmrg_config.ncas}  ({dmrg_config.nelecas} electrons)")
print()
print(f"AEGISS active space   : {len(active_indices)} orbital(s)")
print(f"  Occupied            : {list(active_occ)}")
print(f"  Virtual             : {list(active_virt)}")

## 8. Saved artefacts

When `save_orb=True`, AEGISS writes cube files for:
- Orbitals that passed **entropy screening** (`*_entropy_occ_*.cube`, `*_entropy_virt_*.cube`)
- The **final AEGISS active space** (`*_AEGISS_*.cube`)

When `save_plots=True`, four diagnostic PDFs are written:
- `*_SOE_heatmap.pdf` — bar chart of per-orbital single-orbital entropies
- `*_TH_plot.pdf` — entropy threshold curve across CAS orbitals
- `*_MI_heatmap.pdf` — mutual information matrix (two-orbital entanglement)
- `*_entanglement_diagram.pdf` — entanglement network graph

In [ ]:
written = sorted(os.listdir(output_dir))
print(f"Files in {output_dir}:")
for f in written:
    size = os.path.getsize(os.path.join(output_dir, f))
    print(f"  {f:<50s}  {size:>8,d} bytes")

## 9. Inline entropy bar chart

A quick visual check that the entropy profile matches expectations before
inspecting the saved PDFs.

In [ ]:
try:
    import matplotlib.pyplot as plt
    from aegiss.utils.orbital_indices import get_subspaces_from_cas
    from aegiss.wavefunction.dmrg import run_dmrg

    active_orbs, virt_orbs = get_subspaces_from_cas(
        mf, ncas=dmrg_config.ncas, nelecas=dmrg_config.nelecas
    )
    _energy, soe1, _soe2, _minfo = run_dmrg(
        mf,
        ncas=dmrg_config.ncas,
        nelecas=dmrg_config.nelecas,
        symm_type=dmrg_config.symm_type,
        init_bond_dim=dmrg_config.init_bond_dim,
        verbosity=0,
    )

    all_orbs = list(active_orbs) + list(virt_orbs)
    colors = ["steelblue"] * len(active_orbs) + ["tomato"] * len(virt_orbs)
    threshold = entropy_config.fraction * soe1.max()

    fig, ax = plt.subplots(figsize=(7, 3))
    bars = ax.bar(range(len(soe1)), soe1, color=colors)
    ax.axhline(threshold, ls="--", color="k",
               label=f"entropy threshold ({entropy_config.fraction:.0%} × max)")

    # Highlight the final AEGISS selection
    for idx, orb in enumerate(all_orbs):
        if orb in active_indices:
            bars[idx].set_edgecolor("gold")
            bars[idx].set_linewidth(2.5)

    ax.set_xticks(range(len(soe1)))
    ax.set_xticklabels([f"MO {o}" for o in all_orbs], rotation=45, ha="right")
    ax.set_ylabel("$S_1$")
    ax.set_title("Single-orbital entropies — benzene CAS(6e, 6o)")
    ax.legend()
    plt.tight_layout()
    plt.show()
    print("Gold edges = orbitals selected by AEGISS")
except ImportError:
    print("matplotlib not available — skipping plot.")

## 10. Tuning the entropy threshold

A sweep over `fraction` shows how the size of the active space changes.  The
plateau at six orbitals confirms that the π MOs are well separated from the
rest of the CAS by entropy.

In [ ]:
import tempfile

_tmp = tempfile.mkdtemp()

print(f"{'fraction':>10s}  {'active orbitals':>16s}  {'indices'}")
print("-" * 60)
for frac in (0.01, 0.05, 0.10, 0.20, 0.50):
    idxs = select_aegiss(
        mol=mol,
        mf=mf,
        dmrg_config=dmrg_config,
        entropy_config=EntropyConfig(fraction=frac, fraction_type="r", ranking="subset"),
        projection_config=projection_config,
        output_config=OutputConfig(saving_folder=_tmp, saving_prefix=f"sweep_{frac}",
                                   save_orb=False, save_plots=False),
    )
    print(f"{frac:>10.2f}  {len(idxs):>16d}  {list(idxs)}")

## 11. Multi-group AO labels: π + σ

To capture both the π system **and** the C–H σ bonds, pass two AO groups in
`ao_labels`.  The final active space is the union of orbitals selected by
each group.

Use per-group thresholds when the expected weight scales differ between groups
(here π weights are larger than σ weights on the minimal basis).

In [ ]:
projection_config_multi = ProjectionConfig(
    ao_labels=["C 2pz", "H 1s"],   # two independent AO groups
    basis="minao",
    contribution=[0.3, 0.2],         # per-group absolute thresholds
    contribution_type="a",
)

active_multi = select_aegiss(
    mol=mol,
    mf=mf,
    dmrg_config=dmrg_config,
    entropy_config=entropy_config,
    projection_config=projection_config_multi,
    output_config=OutputConfig(saving_folder=_tmp, saving_prefix="benzene_pi_sigma",
                               save_orb=False, save_plots=False),
)

print(f"π-only selection       : {list(active_indices)}  ({len(active_indices)} orbitals)")
print(f"π + σ selection        : {list(active_multi)}   ({len(active_multi)} orbitals)")

### Multi-shell metals

For systems where two shells should be treated as a **single group** (their
contributions are summed before thresholding), nest them in a list:

```python
# Iron porphyrin — three independent groups
projection_config = ProjectionConfig(
    ao_labels=["Fe 3d", "N 2p", "C 2pz"],
    contribution=[0.4, 0.2, 0.2],
    contribution_type="a",
)

# Cerium complex — 4f + 5d treated as one group
projection_config = ProjectionConfig(
    ao_labels=[["Ce 4f", "Ce 5d"], "N 2p"],
    top_n=[5, 3],
)
```

> **MolTagger** (`aegiss.utils.mol_tagger`) generates these strings
> automatically from an interactive 3D viewer — click atoms, copy labels.

## 12. Top-N mode

When you know the target CAS size in advance, `top_n` guarantees exactly
that many orbitals per group regardless of the weight distribution.
It can be combined with multi-group and per-group lists.

In [ ]:
# Exactly 6 π MOs
active_topn = select_aegiss(
    mol=mol,
    mf=mf,
    dmrg_config=dmrg_config,
    entropy_config=entropy_config,
    projection_config=ProjectionConfig(
        ao_labels=["C 2pz"],
        top_n=6,
    ),
    output_config=OutputConfig(saving_folder=_tmp, saving_prefix="benzene_topn",
                               save_orb=False, save_plots=False),
)
print(f"top_n=6 selection: {list(active_topn)}")

## Summary

| Step | Function / object | Key parameter |
|---|---|---|
| Build molecule | `build_mol` | `atom`, `basis` |
| Mean-field | `run_scf` | `method` (`'RHF'`, `'RKS'`, …) |
| (Optional) Localize | `localize` | `'boys'`, `'pm'`, `'er'`, `'nos'`, `'fnos'` |
| DMRG | `DMRGConfig` | `ncas`, `nelecas`, `symm_type='SZ'` |
| Entropy screening | `EntropyConfig` | `fraction`, `fraction_type`, `ranking` |
| AO projection | `ProjectionConfig` | `ao_labels`, `contribution` / `top_n` |
| Output | `OutputConfig` | `saving_folder`, `save_orb`, `save_plots` |
| Run pipeline | `select_aegiss` | all four configs |

The returned `active_indices` array contains the sorted orbital indices that
define the final active space — ready to pass to a CASSCF or CASCI solver.

```python
ncas_final  = len(active_indices)
nelecas_final = ...   # count electrons in the selected MOs from mf.mo_occ
```